In [ ]:
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
import kagglehub

path = kagglehub.competition_download('clustering-physical-activity-data')

print("Path: ", path)

100%|██████████| 103M/103M [00:03<00:00, 32.8MB/s]

Extracting files...


Path:  /root/.cache/kagglehub/competitions/clustering-physical-activity-data


In [ ]:
# поменять путь
!mv /root/.cache/kagglehub/competitions/clustering-physical-activity-data clustering-physical-activity-data

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('clustering-physical-activity-data/Physical_Activity_Monitoring_unlabeled.csv')
print(df.shape)

(534601, 53)


In [9]:

WINDOW_SIZE = 70
STEP_SIZE = 35

def extract_window_features(window_data):
    """Извлекаем признаки из окна"""
    features = []

    for col in window_data.columns:
        if col not in ['timestamp', 'subject_id']:
            values = window_data[col].dropna().values
            if len(values) > 0:
                features.extend([
                    np.mean(values), np.std(values), np.median(values),
                    np.percentile(values, 25), np.percentile(values, 75),
                    np.max(values) - np.min(values),
                    np.sum(np.abs(np.diff(values))) / len(values) if len(values) > 1 else 0,
                    np.max(values), np.min(values)
                ])
            else:
                features.extend([0] * 9)

    return features

# окна
windows = []
window_row_mapping = []

for subject_id in df['subject_id'].unique():
    subject_data = df[df['subject_id'] == subject_id].sort_values('timestamp')

    for start in range(0, len(subject_data) - WINDOW_SIZE, STEP_SIZE):
        end = start + WINDOW_SIZE
        window = subject_data.iloc[start:end]

        windows.append(extract_window_features(window))
        window_row_mapping.append(subject_data.index[start:end].tolist())


# данные для кластеризации
X = np.array(windows)
X = SimpleImputer(strategy='median').fit_transform(X)
X = StandardScaler().fit_transform(X)

# PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X)
print(f"Размерность: {X_pca.shape[1]}")

# попытка подобрать k
silhouette_scores = []
k_range = range(3, 10)

for k in k_range:
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca)
    if len(set(labels)) > 1:
        score = silhouette_score(X_pca, labels)
        silhouette_scores.append(score)
    else:
        silhouette_scores.append(-1)

optimal_k = k_range[np.argmax(silhouette_scores)]

kmeans = MiniBatchKMeans(n_clusters=optimal_k, random_state=42, n_init=20)
window_labels = kmeans.fit_predict(X_pca)

row_to_cluster = {}
for row_indices, label in zip(window_row_mapping, window_labels):
    for row_idx in row_indices:
        row_to_cluster[row_idx] = label

all_labels = []
for i in range(len(df)):
    if i in row_to_cluster:
        all_labels.append(row_to_cluster[i])
    else:
        all_labels.append(window_labels[0])

all_labels = np.array(all_labels)


def reorder_labels(labels):
    """Перенумеруем кластеры в порядке появления"""
    mapping = {}
    next_id = 1
    new_labels = np.zeros_like(labels)
    for i, label in enumerate(labels):
        if label not in mapping:
            mapping[label] = next_id
            next_id += 1
        new_labels[i] = mapping[label]
    return new_labels.astype(int)

final_labels = reorder_labels(all_labels)

Размерность: 52


In [10]:
with open('submission.csv', 'w', encoding='utf-8') as f:
    f.write('index,activityID\n')
    for i, label in enumerate(final_labels):
        f.write(f'{i},{label}\n')

In [11]:
!kaggle competitions submit -c clustering-physical-activity-data -f submission.csv  -m "FINISH"

100% 4.48M/4.48M [00:01<00:00, 3.66MB/s]
Successfully submitted to Clustering Physical Activity Data